# Reading the tables

In [8]:
%run nb_dq_utils

StatementMeta(, 3850fd7c-6463-458c-94f6-70ad36ea1d72, 24, Finished, Available, Finished, True)

In [9]:
# Run this in the Silver notebook
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.sales")
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.production")
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.hr")

StatementMeta(, 3850fd7c-6463-458c-94f6-70ad36ea1d72, 25, Finished, Available, Finished, False)

DataFrame[]

In [10]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Read Bronze tables
df_header = spark.read.table("lh_adventureworks_bronze.dbo.Sales_SalesOrderHeader")
df_detail = spark.read.table("lh_adventureworks_bronze.dbo.Sales_SalesOrderDetail")
df_customer = spark.read.table("lh_adventureworks_bronze.dbo.Sales_Customer")
df_person = spark.read.table("lh_adventureworks_bronze.dbo.Person_Person")
df_salesperson = spark.read.table("lh_adventureworks_bronze.dbo.Sales_SalesPerson")

StatementMeta(, 3850fd7c-6463-458c-94f6-70ad36ea1d72, 26, Finished, Available, Finished, False)

In [11]:
from pyspark.sql import functions as F
logger = get_logger("sales.sales_orders.dq")

# ── Step 1: Header + Detail ──────────────────────────────────────────
df_sales = df_header.alias("h").join(
    df_detail.alias("d"),
    on="SalesOrderID",
    how="inner"
)

# ── Step 2: Join Customer ─────────────────────────────────────────────
df_customer_clean = df_customer.select(
    F.col("CustomerID").alias("c_CustomerID"),
    F.col("PersonID"),
    F.col("StoreID"),
    F.col("TerritoryID").alias("c_TerritoryID")
)

df_sales = df_sales.join(
    df_customer_clean,
    df_sales["CustomerID"] == df_customer_clean["c_CustomerID"],
    how="left"
).drop("c_CustomerID")

# ── Step 3: Join Person (resolve customer name) ───────────────────────
df_person_clean = df_person.select(
    F.col("BusinessEntityID").alias("p_BusinessEntityID"),
    F.col("FirstName"),
    F.col("LastName")
)

df_sales = df_sales.join(
    df_person_clean,
    df_sales["PersonID"] == df_person_clean["p_BusinessEntityID"],
    how="left"
).drop("p_BusinessEntityID")

# ── Step 4: Join SalesPerson ──────────────────────────────────────────
df_sp_clean = df_salesperson.select(
    F.col("BusinessEntityID").alias("sp_BusinessEntityID"),
    F.col("SalesQuota"),
    F.col("Bonus")
)

df_sales = df_sales.join(
    df_sp_clean,
    df_sales["SalesPersonID"] == df_sp_clean["sp_BusinessEntityID"],
    how="left"
).drop("sp_BusinessEntityID")

# ── Step 5: Build full customer name ──────────────────────────────────
df_sales = df_sales.withColumn(
    "CustomerFullName",
    F.when(
        F.col("FirstName").isNotNull(),
        F.concat_ws(" ", F.col("FirstName"), F.col("LastName"))
    ).otherwise(F.lit("Store/Unknown Customer"))
)

# ── Step 6: Dedup + reject null PKs ──────────────────────────────────
df_sales = df_sales.dropDuplicates(["SalesOrderID", "SalesOrderDetailID"])
df_sales = df_sales.filter(
    F.col("SalesOrderID").isNotNull() & 
    F.col("SalesOrderDetailID").isNotNull()
)

# print("Final row count:", df_sales.count())
# df_sales.printSchema()

StatementMeta(, 3850fd7c-6463-458c-94f6-70ad36ea1d72, 27, Finished, Available, Finished, False)

In [12]:
# ── Step 7: Clean up duplicate and redundant columns ─────────────────

# Drop duplicate rowguid and ModifiedDate from detail side
# Keep header versions (h.), rename for clarity
df_sales = df_sales \
    .drop("c_TerritoryID") \
    .withColumnRenamed("rowguid", "header_rowguid") \
    .withColumnRenamed("ModifiedDate", "header_ModifiedDate")

# The second rowguid and ModifiedDate are from SalesOrderDetail
# Rename those too — but since they're duplicates in the schema,
# we need to select explicitly to separate them
df_sales = df_sales.toDF(*[
    c if i not in [34, 35] else ["detail_rowguid", "detail_ModifiedDate"][i - 34]
    for i, c in enumerate(df_sales.columns)
])

# ── Step 8: Cast short/byte types to integer ─────────────────────────
df_sales = df_sales \
    .withColumn("OrderQty", F.col("OrderQty").cast("integer")) \
    .withColumn("Status", F.col("Status").cast("integer")) \
    .withColumn("RevisionNumber", F.col("RevisionNumber").cast("integer"))

# ── Step 9: Final column selection — only what Silver needs ──────────
df_silver_sales = df_sales.select(
    "SalesOrderID", "SalesOrderDetailID",
    "OrderDate", "DueDate", "ShipDate",
    "SalesOrderNumber", "Status", "OnlineOrderFlag",
    "CustomerID", "CustomerFullName", "PersonID", "StoreID",
    "SalesPersonID", "SalesQuota", "Bonus",
    "TerritoryID",
    "ProductID", "SpecialOfferID",
    "OrderQty", "UnitPrice", "UnitPriceDiscount", "LineTotal",
    "SubTotal", "TaxAmt", "Freight", "TotalDue",
    "PurchaseOrderNumber", "AccountNumber",
    "BillToAddressID", "ShipToAddressID",
    "CreditCardID", "CurrencyRateID",
    "header_rowguid", "header_ModifiedDate"
)
df_silver_sales.cache()
# print("Final column count:", len(df_silver_sales.columns))
# print("Final row count:", df_silver_sales.count())
# df_silver_sales.printSchema()

StatementMeta(, 3850fd7c-6463-458c-94f6-70ad36ea1d72, 28, Finished, Available, Finished, False)

DataFrame[SalesOrderID: int, SalesOrderDetailID: int, OrderDate: timestamp, DueDate: timestamp, ShipDate: timestamp, SalesOrderNumber: string, Status: int, OnlineOrderFlag: boolean, CustomerID: int, CustomerFullName: string, PersonID: int, StoreID: int, SalesPersonID: int, SalesQuota: decimal(19,4), Bonus: decimal(19,4), TerritoryID: int, ProductID: int, SpecialOfferID: int, OrderQty: int, UnitPrice: decimal(19,4), UnitPriceDiscount: decimal(19,4), LineTotal: decimal(38,6), SubTotal: decimal(19,4), TaxAmt: decimal(19,4), Freight: decimal(19,4), TotalDue: decimal(19,4), PurchaseOrderNumber: string, AccountNumber: string, BillToAddressID: int, ShipToAddressID: int, CreditCardID: int, CurrencyRateID: int, header_rowguid: string, header_ModifiedDate: timestamp]

In [13]:
#-----------DQ check------------------------------------
expected_rows = df_detail.count()
actual_rows = df_silver_sales.count()
critical_cols = [
    "SalesOrderID", "SalesOrderDetailID", "CustomerID",
    "TerritoryID", "ProductID", "OrderDate", "LineTotal"
]

unmatched_customers = df_silver_sales.filter(
    F.col("CustomerFullName") == "Store/Unknown Customer"
).count()
unmatched_pct = (unmatched_customers / actual_rows) * 100
 
checks = [
    {
        "name": "Row count",
        "passed": actual_rows == expected_rows,
        "message": f"Expected {expected_rows:,}, got {actual_rows:,} (diff: {actual_rows - expected_rows:+,})"
    },
    *check_null_columns(df_silver_sales, critical_cols),
    check_duplicate_pk(df_silver_sales, ["SalesOrderID", "SalesOrderDetailID"]),
    {
        "name": "Customer join coverage",
        "passed": unmatched_pct < 10,
        "message": f"{unmatched_customers:,} unmatched ({unmatched_pct:.1f}%)"
    },
    check_referential_integrity(df_silver_sales, df_customer, "CustomerID", dim_name="customer"),
]
 
run_dq_checks(checks, logger, "silver.sales_orders")

StatementMeta(, 3850fd7c-6463-458c-94f6-70ad36ea1d72, 29, Finished, Available, Finished, False)

INFO:sales.sales_orders.dq:[DQ] [PASS] Row count — Expected 121,317, got 121,317 (diff: +0)
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — SalesOrderID — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — SalesOrderDetailID — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — CustomerID — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — TerritoryID — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — ProductID — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — OrderDate — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Null check — LineTotal — 0 nulls found
INFO:sales.sales_orders.dq:[DQ] [PASS] Duplicate PK — SalesOrderID, SalesOrderDetailID — 0 duplicates
INFO:sales.sales_orders.dq:[DQ] [PASS] Customer join coverage — 0 unmatched (0.0%)
INFO:sales.sales_orders.dq:[DQ] [PASS] Referential integrity — CustomerID → customer — 0 rows with CustomerID not found in customer
INFO:sales.sales_orders.dq:[DQ

In [14]:
# ── Write to silver Lakehouse ─────────────────────────────────────────
(
    df_silver_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("sales.sales_orders")
)
df_silver_sales.unpersist()
#print("dbo.sales_orders written successfully.")

StatementMeta(, 3850fd7c-6463-458c-94f6-70ad36ea1d72, 30, Finished, Available, Finished, False)

DataFrame[SalesOrderID: int, SalesOrderDetailID: int, OrderDate: timestamp, DueDate: timestamp, ShipDate: timestamp, SalesOrderNumber: string, Status: int, OnlineOrderFlag: boolean, CustomerID: int, CustomerFullName: string, PersonID: int, StoreID: int, SalesPersonID: int, SalesQuota: decimal(19,4), Bonus: decimal(19,4), TerritoryID: int, ProductID: int, SpecialOfferID: int, OrderQty: int, UnitPrice: decimal(19,4), UnitPriceDiscount: decimal(19,4), LineTotal: decimal(38,6), SubTotal: decimal(19,4), TaxAmt: decimal(19,4), Freight: decimal(19,4), TotalDue: decimal(19,4), PurchaseOrderNumber: string, AccountNumber: string, BillToAddressID: int, ShipToAddressID: int, CreditCardID: int, CurrencyRateID: int, header_rowguid: string, header_ModifiedDate: timestamp]

In [15]:
# Quick verify
# df_verify = spark.read.table("dbo.sales_orders")
# print("Verified row count:", df_verify.count())

StatementMeta(, 3850fd7c-6463-458c-94f6-70ad36ea1d72, 31, Finished, Available, Finished, False)